# Fast reproducible SLE comparison with an exact PRAC artifact

This notebook separates **direct experimental comparison** from
**exact native artifact reproduction** and **published literature
context**.

The direct benchmark runs four transition methods under the same
five-party Shamir MP-SPDZ backend, on the same hidden locators, with the
same relation check and the same one-bit public result:

1. **SLE**: apply one key-dependent PRP independently to every database
   and padded-index locator copy.
2. **Oblivious sort/merge**: securely sort database mappings and index
   copies by hidden old locator, propagate the new locator inside each
   equal-key group, then restore index order.
3. **ORAM reindexing**: retain a secret row reference per index copy,
   place new locators in `OptimalORAM`, and perform one hidden lookup
   per padded slot.
4. **Quadratic pointer join**: compare every padded index entry with
   every database locator.

The sort/merge and ORAM implementations are **SOTA-derived
same-backend adaptations**, not exact reproductions of CCS 2020, CGP,
GigaDORAM, PRAC, or another external codebase. Published numbers use
different party counts, cryptography, hardware, outputs, and threat
models; the notebook therefore reports them in a separate table and
never computes a misleading cross-paper speedup.

CPU runtime is required. A Colab GPU does not accelerate MP-SPDZ.

The exact named artifact executed by default is **PRAC/Duoram** at its
pinned `popets-repro` commit. GigaDORAM, Bifrost, and iPrivJoin are not
built in this fast notebook because the previously attempted official
artifacts crashed, timed out, or failed in upstream dependencies.


In [ ]:
# CELL 1. Experiment controls
# Start with MAX_N=16. If it completes, increase to 32 and then 64.
# Radix sorting is much stronger than the old quadratic baseline but
# can still be compilation-heavy because the padded index has M slots.

PAPER_MODE = False
N_VALUES = [8, 16, 32, 64, 128]
MAX_N = 64 if PAPER_MODE else 16

ROOT_BITS = 32
BLOCK_BITS = 4
REPEATS = 10 if PAPER_MODE else 1

RUN_SLE = True
RUN_SORT_MERGE = True
SORT_MERGE_MAX_N = 64
RUN_ORAM_REINDEX = True
ORAM_REINDEX_MAX_N = 64
RUN_POINTER_JOIN = True
POINTER_JOIN_MAX_N = 32

PLAYERS = 5
THRESHOLD = 2
EPOCH = 42

# Exact named SOTA artifact. This executes PRAC's documented native
# Duoram workload and is reported separately from the direct task
# adaptations because its operation and three-party backend differ.
RUN_EXACT_PRAC = True
PRAC_COMMIT = 'ef2b926a0c5c6082cf07cd2f172fcf770c825bb6'
PRAC_DEPTH = 10       # native memory size: 2**10 = 1024
PRAC_ITEMS = 5
PRAC_BUILD_JOBS = 2

# Optional NPZ from the biometric notebook. Required binary array:
# gallery_roots shaped [at least MAX_N, at least ROOT_BITS].
REAL_ROOTS_NPZ = None

ACTIVE_N = [n for n in N_VALUES if n <= MAX_N]
print('Active sizes:', ACTIVE_N)
print('Repetitions:', REPEATS)
print('GPU needed: no')


## What is being compared

Let `N` be the number of enrolled identities, `M` the number of padded
candidate-index slots, and `L=N+M`.

| Method | Transition idea | Dominant generic work |
|---|---|---|
| SLE | Apply the same fresh keyed PRP to every equal locator copy | `(N+M)` PRP evaluations plus relation fingerprint |
| Sort/merge | Build hidden `(old,new)` mappings, obliviously sort/merge, restore order | `N` PRPs plus secure sorting and segmented propagation over `L` rows |
| ORAM reindex | Secret-address lookup through a persistent row reference | `M` ORAM reads plus persistent secret-pointer metadata |
| Pointer join | Test every index locator against every database locator | `N*M` secret equality tests |

A method passes only if `relation_ok=1`. No locator, key, sort order,
old-to-new map, or candidate relation is opened.


In [ ]:
# CELL 2. Install and build MP-SPDZ
import json, os, re, shutil, subprocess, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def run(command, cwd=None, env=None, log_path=None):
    print('$', command)
    started = time.perf_counter()
    process = subprocess.Popen(
        command,
        cwd=cwd,
        env=env,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    lines = []
    for line in process.stdout:
        print(line, end='')
        lines.append(line)
    returncode = process.wait()
    output = ''.join(lines)
    elapsed = time.perf_counter() - started
    print(f'elapsed={elapsed:.2f}s returncode={returncode}')
    if log_path is not None:
        Path(log_path).write_text(output, encoding='utf-8')
    if returncode:
        raise RuntimeError(command)
    return output, elapsed

# These two repair commands prevent the interrupted-dpkg error that
# occasionally appears after a Colab runtime restart.
run('DEBIAN_FRONTEND=noninteractive dpkg --configure -a')
run('DEBIAN_FRONTEND=noninteractive apt-get -qq --fix-broken install -y')
run(
    'apt-get -qq update && DEBIAN_FRONTEND=noninteractive '
    'apt-get -qq install -y automake build-essential clang cmake git '
    'libboost-dev libboost-filesystem-dev libboost-iostreams-dev '
    'libboost-thread-dev libgmp-dev libntl-dev libsodium-dev '
    'libssl-dev libtool libbsd-dev numactl '
    'libboost-context-dev libboost-chrono-dev libboost-system-dev'
)

if not Path('/content/MP-SPDZ').exists():
    run(
        'git clone --depth 1 '
        'https://github.com/data61/MP-SPDZ.git /content/MP-SPDZ'
    )
run('make -j2 setup', cwd='/content/MP-SPDZ')
run('make -j2 shamir-party.x', cwd='/content/MP-SPDZ')
run('Scripts/setup-ssl.sh 5', cwd='/content/MP-SPDZ')

commit, _ = run('git rev-parse HEAD', cwd='/content/MP-SPDZ')
MP_SPDZ_COMMIT = commit.strip().splitlines()[-1]
print('MP-SPDZ commit:', MP_SPDZ_COMMIT)


## Embedded reviewed source files

All protocol and input-generation files are embedded below. Only this
`.ipynb` needs to be uploaded to Colab.


In [ ]:
# This cell reconstructs one reviewed project file inside Colab.
from pathlib import Path
target = Path('/content/sle_sota_project/Programs/Source/sle_transition_optimized.mpc')
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""\nRound-optimized MPC implementation of one SLE epoch transition.\n\nThe semantics match sle_transition.mpc:\n  * five parties jointly contribute the epoch key;\n  * database and index copies evolve without opening a locator;\n  * the hidden (locator, block, bucket) relation is checked twice;\n  * only relation_ok is opened and state selection is atomic.\n\nThe implementation changes two performance details only. First, all locator\ncopies are evaluated as register vectors, so independent PRP instances are\nbatched across the same 73 sequential rounds. Second, multiset fingerprints\nuse a balanced product tree instead of one long dependency chain.\n\nCompile-time arguments:\n    sle_transition_optimized <n_records> <root_bits> <block_bits>\n                             <bucket_capacity> <epoch> <fault_mode>\n"""\n\nfrom Compiler.library import tree_reduce\n\nprogram.use_edabit(True)\nprogram.set_bit_length(64)\n\nif len(program.args) != 7:\n    raise CompilerError(\n        "Usage: sle_transition_optimized <n_records> <root_bits> "\n        "<block_bits> <bucket_capacity> <epoch> <fault_mode>"\n    )\n\nN_RECORDS = int(program.args[1])\nROOT_BITS = int(program.args[2])\nBLOCK_BITS = int(program.args[3])\nBUCKET_CAPACITY = int(program.args[4])\nNEW_EPOCH = int(program.args[5])\nFAULT_MODE = int(program.args[6])\n\nif N_RECORDS < 2:\n    raise CompilerError("n_records must be at least two")\nif ROOT_BITS % BLOCK_BITS != 0:\n    raise CompilerError("root_bits must be divisible by block_bits")\nif FAULT_MODE not in (0, 1, 2, 3):\n    raise CompilerError("fault_mode must be 0, 1, 2, or 3")\n\nN_BLOCKS = ROOT_BITS // BLOCK_BITS\nBUCKET_VALUES = 2 ** BLOCK_BITS\nN_BUCKETS = N_BLOCKS * BUCKET_VALUES\nN_INDEX_SLOTS = N_BUCKETS * BUCKET_CAPACITY\nMIMC_ROUNDS = 73\n\n\nold_db_locators = sint.Array(N_RECORDS)\nroots = sint.Matrix(N_RECORDS, ROOT_BITS)\nold_index_locators = sint.Array(N_INDEX_SLOTS)\nindex_valid = sint.Array(N_INDEX_SLOTS)\n\nold_db_locators.input_from(0)\nroots.input_from(0)\nold_index_locators.input_from(0)\nindex_valid.input_from(0)\n\n# The complete epoch key is never supplied by one party.\nepoch_key = sint(0)\nfor party in range(5):\n    epoch_key += sint.get_input_from(party)\n\nMIMC_CONSTANTS = [\n    int.from_bytes(\n        __import__("hashlib").sha256(\n            ("SLE-MiMC-round-%d" % round_no).encode()\n        ).digest()[:8],\n        "big",\n    )\n    for round_no in range(MIMC_ROUNDS)\n]\n\n\ndef secure_locator_prp_vector(values):\n    """Evaluate independent locator transitions in one vectorized circuit."""\n\n    x = values + NEW_EPOCH\n    for constant in MIMC_CONSTANTS:\n        x = (x + epoch_key + constant) ** 3\n    return x + epoch_key\n\n\ndef root_bucket_value(root, block):\n    value = sint(0)\n    for bit in range(BLOCK_BITS):\n        value += root[block * BLOCK_BITS + bit] * (2 ** bit)\n    return value\n\n\ndef product_tree(factors):\n    """Multiply factors with logarithmic dependency depth."""\n\n    if not factors:\n        return sint(1)\n    return tree_reduce(lambda left, right: left * right, factors)\n\n\nstart_timer(3)\n\nnew_db_locators = sint.Array(N_RECORDS)\nnew_index_locators = sint.Array(N_INDEX_SLOTS)\nnew_index_valid = sint.Array(N_INDEX_SLOTS)\n\nnew_db_locators.assign_vector(\n    secure_locator_prp_vector(old_db_locators.get_vector())\n)\nnew_index_locators.assign_vector(\n    secure_locator_prp_vector(old_index_locators.get_vector())\n)\nnew_index_valid.assign_vector(index_valid.get_vector())\n\n# These controlled faults test the relation gate. They are public test modes,\n# not a claim of active security for the passive Shamir backend.\nif FAULT_MODE == 1:\n    new_index_valid[0] = sint(0)\nelif FAULT_MODE == 2:\n    new_index_locators[0] = new_index_locators[0] + 1\nelif FAULT_MODE == 3:\n    temporary = new_index_locators[0]\n    new_index_locators[0] = new_index_locators[BUCKET_CAPACITY]\n    new_index_locators[BUCKET_CAPACITY] = temporary\n\n\n# Two public challenges are sampled after candidate values have been formed.\nchallenge_1 = sint.get_random_int(60).reveal() + 101\nchallenge_2 = sint.get_random_int(60).reveal() + 211\nalpha_1 = cint(1_000_003)\nbeta_1 = cint(1_000_033)\nalpha_2 = cint(1_000_037)\nbeta_2 = cint(1_000_081)\n\nexpected_factors_1 = []\nexpected_factors_2 = []\nfor row in range(N_RECORDS):\n    locator = new_db_locators[row]\n    for block in range(N_BLOCKS):\n        bucket = root_bucket_value(roots[row], block)\n        element_1 = locator + alpha_1 * block + beta_1 * bucket\n        element_2 = locator + alpha_2 * block + beta_2 * bucket\n        expected_factors_1.append(challenge_1 - element_1)\n        expected_factors_2.append(challenge_2 - element_2)\n\nactual_factors_1 = []\nactual_factors_2 = []\nfor position in range(N_INDEX_SLOTS):\n    bucket_address = position // BUCKET_CAPACITY\n    block = bucket_address // BUCKET_VALUES\n    bucket = bucket_address % BUCKET_VALUES\n    locator = new_index_locators[position]\n    valid = new_index_valid[position]\n\n    element_1 = locator + alpha_1 * block + beta_1 * bucket\n    element_2 = locator + alpha_2 * block + beta_2 * bucket\n    actual_factors_1.append(\n        valid * (challenge_1 - element_1) + (1 - valid)\n    )\n    actual_factors_2.append(\n        valid * (challenge_2 - element_2) + (1 - valid)\n    )\n\nexpected_1 = product_tree(expected_factors_1)\nexpected_2 = product_tree(expected_factors_2)\nactual_1 = product_tree(actual_factors_1)\nactual_2 = product_tree(actual_factors_2)\nrelation_ok = (expected_1 == actual_1) * (expected_2 == actual_2)\n\n\n# A failed transition preserves the old shared state.\ncommitted_db_locators = sint.Array(N_RECORDS)\ncommitted_index_locators = sint.Array(N_INDEX_SLOTS)\ncommitted_index_valid = sint.Array(N_INDEX_SLOTS)\n\nfor row in range(N_RECORDS):\n    committed_db_locators[row] = relation_ok.if_else(\n        new_db_locators[row], old_db_locators[row]\n    )\n\nfor position in range(N_INDEX_SLOTS):\n    committed_index_locators[position] = relation_ok.if_else(\n        new_index_locators[position], old_index_locators[position]\n    )\n    committed_index_valid[position] = relation_ok.if_else(\n        new_index_valid[position], index_valid[position]\n    )\n\ncommitted_db_locators.write_to_file()\ncommitted_index_locators.write_to_file()\ncommitted_index_valid.write_to_file()\n\nopened_relation_ok = relation_ok.reveal()\nstop_timer(3)\n\nprint_ln(\n    "SLE_MPC_RESULT mode=transition_optimized epoch=%s relation_ok=%s fault=%s",\n    NEW_EPOCH,\n    opened_relation_ok,\n    FAULT_MODE,\n)\nprint_ln(\n    "SLE_MPC_PRIVATE key=not_opened locators=not_opened "\n    "mapping=not_materialized"\n)\n', encoding='utf-8')
print('Wrote:', target)


In [ ]:
# This cell reconstructs one reviewed project file inside Colab.
from pathlib import Path
target = Path('/content/sle_sota_project/Programs/Source/sle_sort_merge_transition.mpc')
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""\nOblivious radix-sort/merge baseline for an SLE epoch transition.\n\nThis program is a stronger same-backend baseline than the quadratic secret\npointer join. It treats transition as a generic hidden-key translation:\n\n  1. evolve the N database locators;\n  2. concatenate database mapping rows and M padded index rows;\n  3. obliviously radix-sort the secret rows by old locator;\n  4. propagate the corresponding new locator inside each equal-key segment;\n  5. obliviously restore the original index order;\n  6. run the same hidden relation check and atomic commit used by SLE.\n\nIt is deliberately called a SOTA-inspired baseline rather than a reproduction\nof any one published join or shuffle system. The code runs in the same\nfive-party Shamir MP-SPDZ environment as SLE, which makes runtime,\ncommunication, and round measurements directly comparable.\n\nCompile-time arguments:\n    sle_sort_merge_transition <n_records> <root_bits> <block_bits>\n                              <bucket_capacity> <epoch>\n"""\n\nfrom Compiler import sorting\nfrom Compiler.library import tree_reduce\n\nprogram.use_edabit(True)\nprogram.set_bit_length(64)\n\nif len(program.args) != 6:\n    raise CompilerError(\n        "Usage: sle_sort_merge_transition <n_records> <root_bits> "\n        "<block_bits> <bucket_capacity> <epoch>"\n    )\n\nN_RECORDS = int(program.args[1])\nROOT_BITS = int(program.args[2])\nBLOCK_BITS = int(program.args[3])\nBUCKET_CAPACITY = int(program.args[4])\nNEW_EPOCH = int(program.args[5])\n\nif N_RECORDS < 2:\n    raise CompilerError("n_records must be at least two")\nif ROOT_BITS % BLOCK_BITS != 0:\n    raise CompilerError("root_bits must be divisible by block_bits")\n\nN_BLOCKS = ROOT_BITS // BLOCK_BITS\nBUCKET_VALUES = 2 ** BLOCK_BITS\nN_BUCKETS = N_BLOCKS * BUCKET_VALUES\nN_INDEX_SLOTS = N_BUCKETS * BUCKET_CAPACITY\nN_COMBINED = N_RECORDS + N_INDEX_SLOTS\nMIMC_ROUNDS = 73\n\n# Input locators are generated as positive 31-bit integers. One extra bit is\n# used to place database rows before index rows for equal old locators.\nJOIN_KEY_BITS = 32\nRESTORE_KEY_BITS = (2 * N_COMBINED - 1).bit_length()\n\n\nold_db_locators = sint.Array(N_RECORDS)\nroots = sint.Matrix(N_RECORDS, ROOT_BITS)\nold_index_locators = sint.Array(N_INDEX_SLOTS)\nindex_valid = sint.Array(N_INDEX_SLOTS)\n\nold_db_locators.input_from(0)\nroots.input_from(0)\nold_index_locators.input_from(0)\nindex_valid.input_from(0)\n\n# The full epoch key remains distributed across five computing parties.\nepoch_key = sint(0)\nfor party in range(5):\n    epoch_key += sint.get_input_from(party)\n\nMIMC_CONSTANTS = [\n    int.from_bytes(\n        __import__("hashlib").sha256(\n            ("SLE-MiMC-round-%d" % round_no).encode()\n        ).digest()[:8],\n        "big",\n    )\n    for round_no in range(MIMC_ROUNDS)\n]\n\n\ndef secure_locator_prp_vector(values):\n    """Apply the same MPC-friendly epoch PRP used by SLE."""\n\n    x = values + NEW_EPOCH\n    for constant in MIMC_CONSTANTS:\n        x = (x + epoch_key + constant) ** 3\n    return x + epoch_key\n\n\ndef root_bucket_value(root, block):\n    value = sint(0)\n    for bit in range(BLOCK_BITS):\n        value += root[block * BLOCK_BITS + bit] * (2 ** bit)\n    return value\n\n\ndef product_tree(factors):\n    if not factors:\n        return sint(1)\n    return tree_reduce(lambda left, right: left * right, factors)\n\n\nstart_timer(5)\n\n# Step 1: evolve the canonical database locators.\nnew_db_locators = sint.Array(N_RECORDS)\nnew_db_locators.assign_vector(\n    secure_locator_prp_vector(old_db_locators.get_vector())\n)\n\n# Record columns:\n#   0 old locator, 1 translated/new locator, 2 has mapping source,\n#   3 row type (0 database, 1 index), 4 original position, 5 index valid.\nrecords = sint.Matrix(N_COMBINED, 6)\njoin_keys = sint.Array(N_COMBINED)\n\nfor row in range(N_RECORDS):\n    records[row][0] = old_db_locators[row]\n    records[row][1] = new_db_locators[row]\n    records[row][2] = sint(1)\n    records[row][3] = sint(0)\n    records[row][4] = sint(N_INDEX_SLOTS + row)\n    records[row][5] = sint(0)\n    join_keys[row] = old_db_locators[row] * 2\n\nfor position in range(N_INDEX_SLOTS):\n    target = N_RECORDS + position\n    records[target][0] = old_index_locators[position]\n    records[target][1] = sint(0)\n    records[target][2] = sint(0)\n    records[target][3] = sint(1)\n    records[target][4] = sint(position)\n    records[target][5] = index_valid[position]\n    # Database row type 0 sorts before index row type 1 on equal keys.\n    join_keys[target] = old_index_locators[position] * 2 + 1\n\n# Step 2: hidden-key sort. MP-SPDZ radix_sort does not reveal the sorted keys\n# or payload order; its internal revealed permutations are randomized.\nsorting.radix_sort(\n    join_keys,\n    records,\n    n_bits=JOIN_KEY_BITS,\n    signed=False,\n)\n\n# Step 3: segmented parallel prefix propagation. The database mapping is the\n# first row in every real equal-locator segment. Doubling the stride propagates\n# it to all copies in logarithmic dependency depth.\ngroup_keys = sint.Array(N_COMBINED)\ncarried_values = sint.Array(N_COMBINED)\nhas_source = sint.Array(N_COMBINED)\nfor position in range(N_COMBINED):\n    group_keys[position] = records[position][0]\n    carried_values[position] = records[position][1]\n    has_source[position] = records[position][2]\n\nstride = 1\nwhile stride < N_COMBINED:\n    next_values = sint.Array(N_COMBINED)\n    next_has_source = sint.Array(N_COMBINED)\n\n    for position in range(N_COMBINED):\n        if position < stride:\n            next_values[position] = carried_values[position]\n            next_has_source[position] = has_source[position]\n        else:\n            same_segment = (\n                group_keys[position] == group_keys[position - stride]\n            )\n            take_left = (\n                same_segment\n                * has_source[position - stride]\n                * (1 - has_source[position])\n            )\n            next_values[position] = take_left.if_else(\n                carried_values[position - stride],\n                carried_values[position],\n            )\n            next_has_source[position] = has_source[position] + take_left\n\n    carried_values = next_values\n    has_source = next_has_source\n    stride *= 2\n\n# Valid index rows must receive exactly one database mapping. Padded dummies\n# must receive none and are re-randomized instead of being linked to a record.\njoin_checks = []\nfor position in range(N_COMBINED):\n    is_index = records[position][3]\n    valid = records[position][5]\n    source_found = has_source[position]\n    index_row_ok = (\n        valid * source_found\n        + (1 - valid) * (1 - source_found)\n    )\n    join_checks.append(\n        is_index * index_row_ok + (1 - is_index)\n    )\n\n    fresh_dummy = sint.get_random_int(60)\n    translated_index_value = valid.if_else(\n        carried_values[position], fresh_dummy\n    )\n    records[position][1] = is_index.if_else(\n        translated_index_value,\n        records[position][1],\n    )\n\njoin_ok = product_tree(join_checks)\n\n# Step 4: obliviously restore original order, placing index rows first.\nrestore_keys = sint.Array(N_COMBINED)\nfor position in range(N_COMBINED):\n    is_index = records[position][3]\n    restore_keys[position] = (\n        (1 - is_index) * N_COMBINED + records[position][4]\n    )\n\nsorting.radix_sort(\n    restore_keys,\n    records,\n    n_bits=RESTORE_KEY_BITS,\n    signed=False,\n)\n\nnew_index_locators = sint.Array(N_INDEX_SLOTS)\nnew_index_valid = sint.Array(N_INDEX_SLOTS)\nfor position in range(N_INDEX_SLOTS):\n    new_index_locators[position] = records[position][1]\n    new_index_valid[position] = records[position][5]\n\n# Same two-challenge hidden multiset relation gate used by SLE.\nchallenge_1 = sint.get_random_int(60).reveal() + 101\nchallenge_2 = sint.get_random_int(60).reveal() + 211\nalpha_1 = cint(1_000_003)\nbeta_1 = cint(1_000_033)\nalpha_2 = cint(1_000_037)\nbeta_2 = cint(1_000_081)\n\nexpected_factors_1 = []\nexpected_factors_2 = []\nfor row in range(N_RECORDS):\n    locator = new_db_locators[row]\n    for block in range(N_BLOCKS):\n        bucket = root_bucket_value(roots[row], block)\n        expected_factors_1.append(\n            challenge_1 - (locator + alpha_1 * block + beta_1 * bucket)\n        )\n        expected_factors_2.append(\n            challenge_2 - (locator + alpha_2 * block + beta_2 * bucket)\n        )\n\nactual_factors_1 = []\nactual_factors_2 = []\nfor position in range(N_INDEX_SLOTS):\n    bucket_address = position // BUCKET_CAPACITY\n    block = bucket_address // BUCKET_VALUES\n    bucket = bucket_address % BUCKET_VALUES\n    locator = new_index_locators[position]\n    valid = new_index_valid[position]\n    actual_factors_1.append(\n        valid * (\n            challenge_1 - (locator + alpha_1 * block + beta_1 * bucket)\n        ) + (1 - valid)\n    )\n    actual_factors_2.append(\n        valid * (\n            challenge_2 - (locator + alpha_2 * block + beta_2 * bucket)\n        ) + (1 - valid)\n    )\n\nrelation_ok = (\n    join_ok\n    * (product_tree(expected_factors_1) == product_tree(actual_factors_1))\n    * (product_tree(expected_factors_2) == product_tree(actual_factors_2))\n)\n\n# Failed transitions preserve the previous shared state.\ncommitted_db_locators = sint.Array(N_RECORDS)\ncommitted_index_locators = sint.Array(N_INDEX_SLOTS)\ncommitted_index_valid = sint.Array(N_INDEX_SLOTS)\n\nfor row in range(N_RECORDS):\n    committed_db_locators[row] = relation_ok.if_else(\n        new_db_locators[row], old_db_locators[row]\n    )\n\nfor position in range(N_INDEX_SLOTS):\n    committed_index_locators[position] = relation_ok.if_else(\n        new_index_locators[position], old_index_locators[position]\n    )\n    committed_index_valid[position] = relation_ok.if_else(\n        new_index_valid[position], index_valid[position]\n    )\n\ncommitted_db_locators.write_to_file()\ncommitted_index_locators.write_to_file()\ncommitted_index_valid.write_to_file()\n\nopened_relation_ok = relation_ok.reveal()\nstop_timer(5)\n\nprint_ln(\n    "SLE_MPC_RESULT mode=sort_merge epoch=%s relation_ok=%s",\n    NEW_EPOCH,\n    opened_relation_ok,\n)\nprint_ln(\n    "SLE_MPC_PRIVATE key=not_opened locators=not_opened "\n    "sort_order=not_opened pointer_map=not_opened"\n)\n', encoding='utf-8')
print('Wrote:', target)


In [ ]:
# This cell reconstructs one reviewed project file inside Colab.
from pathlib import Path
target = Path('/content/sle_sota_project/Programs/Source/sle_pointer_join_transition.mpc')
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""\nGeneric secret pointer-translation baseline for an SLE epoch transition.\n\nThe baseline gives SLE a meaningful competitor under the same five-party\nShamir execution and the same opened output. It first evolves the N database\nlocators. Every one of the M padded index entries is then joined against all N\nold database locators with secret equality tests in order to recover the\ncorresponding new locator. No pointer map is opened, but O(N*M) secure equality\nwork is required.\n\nThis is deliberately not presented as a novel protocol. It is an executable\ngeneric MPC baseline for the pointer-translation step that SLE avoids.\n\nCompile-time arguments:\n    sle_pointer_join_transition <n_records> <root_bits> <block_bits>\n                                <bucket_capacity> <epoch>\n"""\n\nfrom Compiler.library import tree_reduce\n\nprogram.use_edabit(True)\nprogram.set_bit_length(64)\n\nif len(program.args) != 6:\n    raise CompilerError(\n        "Usage: sle_pointer_join_transition <n_records> <root_bits> "\n        "<block_bits> <bucket_capacity> <epoch>"\n    )\n\nN_RECORDS = int(program.args[1])\nROOT_BITS = int(program.args[2])\nBLOCK_BITS = int(program.args[3])\nBUCKET_CAPACITY = int(program.args[4])\nNEW_EPOCH = int(program.args[5])\n\nif N_RECORDS < 2:\n    raise CompilerError("n_records must be at least two")\nif ROOT_BITS % BLOCK_BITS != 0:\n    raise CompilerError("root_bits must be divisible by block_bits")\n\nN_BLOCKS = ROOT_BITS // BLOCK_BITS\nBUCKET_VALUES = 2 ** BLOCK_BITS\nN_BUCKETS = N_BLOCKS * BUCKET_VALUES\nN_INDEX_SLOTS = N_BUCKETS * BUCKET_CAPACITY\nMIMC_ROUNDS = 73\n\n\nold_db_locators = sint.Array(N_RECORDS)\nroots = sint.Matrix(N_RECORDS, ROOT_BITS)\nold_index_locators = sint.Array(N_INDEX_SLOTS)\nindex_valid = sint.Array(N_INDEX_SLOTS)\n\nold_db_locators.input_from(0)\nroots.input_from(0)\nold_index_locators.input_from(0)\nindex_valid.input_from(0)\n\nepoch_key = sint(0)\nfor party in range(5):\n    epoch_key += sint.get_input_from(party)\n\nMIMC_CONSTANTS = [\n    int.from_bytes(\n        __import__("hashlib").sha256(\n            ("SLE-MiMC-round-%d" % round_no).encode()\n        ).digest()[:8],\n        "big",\n    )\n    for round_no in range(MIMC_ROUNDS)\n]\n\n\ndef secure_locator_prp_vector(values):\n    x = values + NEW_EPOCH\n    for constant in MIMC_CONSTANTS:\n        x = (x + epoch_key + constant) ** 3\n    return x + epoch_key\n\n\ndef root_bucket_value(root, block):\n    value = sint(0)\n    for bit in range(BLOCK_BITS):\n        value += root[block * BLOCK_BITS + bit] * (2 ** bit)\n    return value\n\n\ndef product_tree(factors):\n    if not factors:\n        return sint(1)\n    return tree_reduce(lambda left, right: left * right, factors)\n\n\nstart_timer(4)\n\n# Database transition is identical to optimized SLE.\nnew_db_locators = sint.Array(N_RECORDS)\nnew_db_locators.assign_vector(\n    secure_locator_prp_vector(old_db_locators.get_vector())\n)\n\n# Generic pointer translation. Each valid index copy must discover which old\n# database locator it refers to before selecting the corresponding new value.\nnew_index_locators = sint.Array(N_INDEX_SLOTS)\nnew_index_valid = sint.Array(N_INDEX_SLOTS)\njoin_checks = []\n\nfor position in range(N_INDEX_SLOTS):\n    valid = index_valid[position]\n    translated = sint(0)\n    number_of_matches = sint(0)\n\n    for row in range(N_RECORDS):\n        is_match = old_index_locators[position] == old_db_locators[row]\n        translated += is_match * new_db_locators[row]\n        number_of_matches += is_match\n\n    # Dummy slots receive fresh secret values. For valid entries the equality\n    # join must find exactly one database record; dummies must find none.\n    dummy_locator = sint.get_random_int(60)\n    new_index_locators[position] = valid.if_else(\n        translated, dummy_locator\n    )\n    new_index_valid[position] = valid\n    join_checks.append(\n        valid * (number_of_matches == 1)\n        + (1 - valid) * (number_of_matches == 0)\n    )\n\njoin_ok = product_tree(join_checks)\n\n# Use the same hidden relation check as SLE so both transitions have the same\n# integrity output and commit semantics.\nchallenge_1 = sint.get_random_int(60).reveal() + 101\nchallenge_2 = sint.get_random_int(60).reveal() + 211\nalpha_1 = cint(1_000_003)\nbeta_1 = cint(1_000_033)\nalpha_2 = cint(1_000_037)\nbeta_2 = cint(1_000_081)\n\nexpected_factors_1 = []\nexpected_factors_2 = []\nfor row in range(N_RECORDS):\n    locator = new_db_locators[row]\n    for block in range(N_BLOCKS):\n        bucket = root_bucket_value(roots[row], block)\n        expected_factors_1.append(\n            challenge_1 - (locator + alpha_1 * block + beta_1 * bucket)\n        )\n        expected_factors_2.append(\n            challenge_2 - (locator + alpha_2 * block + beta_2 * bucket)\n        )\n\nactual_factors_1 = []\nactual_factors_2 = []\nfor position in range(N_INDEX_SLOTS):\n    bucket_address = position // BUCKET_CAPACITY\n    block = bucket_address // BUCKET_VALUES\n    bucket = bucket_address % BUCKET_VALUES\n    locator = new_index_locators[position]\n    valid = new_index_valid[position]\n    actual_factors_1.append(\n        valid * (\n            challenge_1 - (locator + alpha_1 * block + beta_1 * bucket)\n        ) + (1 - valid)\n    )\n    actual_factors_2.append(\n        valid * (\n            challenge_2 - (locator + alpha_2 * block + beta_2 * bucket)\n        ) + (1 - valid)\n    )\n\nrelation_ok = (\n    join_ok\n    * (product_tree(expected_factors_1) == product_tree(actual_factors_1))\n    * (product_tree(expected_factors_2) == product_tree(actual_factors_2))\n)\n\ncommitted_db_locators = sint.Array(N_RECORDS)\ncommitted_index_locators = sint.Array(N_INDEX_SLOTS)\ncommitted_index_valid = sint.Array(N_INDEX_SLOTS)\n\nfor row in range(N_RECORDS):\n    committed_db_locators[row] = relation_ok.if_else(\n        new_db_locators[row], old_db_locators[row]\n    )\n\nfor position in range(N_INDEX_SLOTS):\n    committed_index_locators[position] = relation_ok.if_else(\n        new_index_locators[position], old_index_locators[position]\n    )\n    committed_index_valid[position] = relation_ok.if_else(\n        new_index_valid[position], index_valid[position]\n    )\n\ncommitted_db_locators.write_to_file()\ncommitted_index_locators.write_to_file()\ncommitted_index_valid.write_to_file()\n\nopened_relation_ok = relation_ok.reveal()\nstop_timer(4)\n\nprint_ln(\n    "SLE_MPC_RESULT mode=pointer_join epoch=%s relation_ok=%s",\n    NEW_EPOCH,\n    opened_relation_ok,\n)\nprint_ln(\n    "SLE_MPC_PRIVATE key=not_opened locators=not_opened "\n    "pointer_map=not_opened"\n)\n', encoding='utf-8')
print('Wrote:', target)


In [ ]:
# This cell reconstructs one reviewed project file inside Colab.
from pathlib import Path
target = Path('/content/sle_sota_project/Programs/Source/sle_oram_reindex_transition.mpc')
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('"""\nORAM-based private reindexing baseline for an SLE epoch transition.\n\nThis baseline adapts a standard private-memory design to the exact coupled\ndatabase/index transition evaluated by SLE.  Each padded index slot carries a\npersistent secret-shared row reference.  Fresh database locators are stored in\nMP-SPDZ OptimalORAM and every slot performs one secret-address lookup.  The\naccess pattern and row references remain hidden, but the design requires\npersistent pointer metadata that SLE does not materialize.\n\nThe database transition, relation fingerprint, public output, and atomic\ncommit semantics are identical to the other same-backend baselines.\n\nCompile-time arguments:\n    sle_oram_reindex_transition <n_records> <root_bits> <block_bits>\n                                <bucket_capacity> <epoch>\n"""\n\nfrom Compiler.library import tree_reduce\nfrom oram import OptimalORAM\n\nprogram.use_edabit(True)\nprogram.set_bit_length(64)\n\nif len(program.args) != 6:\n    raise CompilerError(\n        "Usage: sle_oram_reindex_transition <n_records> <root_bits> "\n        "<block_bits> <bucket_capacity> <epoch>"\n    )\n\nN_RECORDS = int(program.args[1])\nROOT_BITS = int(program.args[2])\nBLOCK_BITS = int(program.args[3])\nBUCKET_CAPACITY = int(program.args[4])\nNEW_EPOCH = int(program.args[5])\n\nif N_RECORDS < 2:\n    raise CompilerError("n_records must be at least two")\nif ROOT_BITS % BLOCK_BITS != 0:\n    raise CompilerError("root_bits must be divisible by block_bits")\n\nN_BLOCKS = ROOT_BITS // BLOCK_BITS\nBUCKET_VALUES = 2 ** BLOCK_BITS\nN_BUCKETS = N_BLOCKS * BUCKET_VALUES\nN_INDEX_SLOTS = N_BUCKETS * BUCKET_CAPACITY\nMIMC_ROUNDS = 73\n\n\nold_db_locators = sint.Array(N_RECORDS)\nroots = sint.Matrix(N_RECORDS, ROOT_BITS)\nold_index_locators = sint.Array(N_INDEX_SLOTS)\nindex_valid = sint.Array(N_INDEX_SLOTS)\n\nold_db_locators.input_from(0)\nroots.input_from(0)\nold_index_locators.input_from(0)\nindex_valid.input_from(0)\n\nepoch_key = sint(0)\nfor party in range(5):\n    epoch_key += sint.get_input_from(party)\n\n# These secret references are the extra state required by the ORAM baseline.\n# Invalid padded slots may contain the dummy address N_RECORDS.\nindex_row_references = sint.Array(N_INDEX_SLOTS)\nindex_row_references.input_from(0)\n\nMIMC_CONSTANTS = [\n    int.from_bytes(\n        __import__("hashlib").sha256(\n            ("SLE-MiMC-round-%d" % round_no).encode()\n        ).digest()[:8],\n        "big",\n    )\n    for round_no in range(MIMC_ROUNDS)\n]\n\n\ndef secure_locator_prp_vector(values):\n    x = values + NEW_EPOCH\n    for constant in MIMC_CONSTANTS:\n        x = (x + epoch_key + constant) ** 3\n    return x + epoch_key\n\n\ndef root_bucket_value(root, block):\n    value = sint(0)\n    for bit in range(BLOCK_BITS):\n        value += root[block * BLOCK_BITS + bit] * (2 ** bit)\n    return value\n\n\ndef product_tree(factors):\n    if not factors:\n        return sint(1)\n    return tree_reduce(lambda left, right: left * right, factors)\n\n\nstart_timer(5)\n\nnew_db_locators = sint.Array(N_RECORDS)\nnew_db_locators.assign_vector(\n    secure_locator_prp_vector(old_db_locators.get_vector())\n)\n\n# One extra dummy row keeps every secret address in range.  Its value is never\n# committed because invalid slots are masked below.\nlocator_oram = OptimalORAM(N_RECORDS + 1, entry_size=(64,))\nfor row in range(N_RECORDS):\n    locator_oram[row] = new_db_locators[row]\nlocator_oram[N_RECORDS] = sint.get_random_int(60)\n\ndummy_candidates = sint.Array(N_INDEX_SLOTS)\ndummy_candidates.assign_vector(\n    secure_locator_prp_vector(old_index_locators.get_vector())\n)\n\nnew_index_locators = sint.Array(N_INDEX_SLOTS)\nnew_index_valid = sint.Array(N_INDEX_SLOTS)\n\nfor position in range(N_INDEX_SLOTS):\n    valid = index_valid[position]\n    translated = locator_oram[index_row_references[position]]\n    new_index_locators[position] = valid.if_else(\n        translated, dummy_candidates[position]\n    )\n    new_index_valid[position] = valid\n\n# The same two-challenge hidden multiset check used by SLE prevents a method\n# from appearing faster by omitting verification or rollback semantics.\nchallenge_1 = sint.get_random_int(60).reveal() + 101\nchallenge_2 = sint.get_random_int(60).reveal() + 211\nalpha_1 = cint(1_000_003)\nbeta_1 = cint(1_000_033)\nalpha_2 = cint(1_000_037)\nbeta_2 = cint(1_000_081)\n\nexpected_factors_1 = []\nexpected_factors_2 = []\nfor row in range(N_RECORDS):\n    locator = new_db_locators[row]\n    for block in range(N_BLOCKS):\n        bucket = root_bucket_value(roots[row], block)\n        expected_factors_1.append(\n            challenge_1 - (locator + alpha_1 * block + beta_1 * bucket)\n        )\n        expected_factors_2.append(\n            challenge_2 - (locator + alpha_2 * block + beta_2 * bucket)\n        )\n\nactual_factors_1 = []\nactual_factors_2 = []\nfor position in range(N_INDEX_SLOTS):\n    bucket_address = position // BUCKET_CAPACITY\n    block = bucket_address // BUCKET_VALUES\n    bucket = bucket_address % BUCKET_VALUES\n    locator = new_index_locators[position]\n    valid = new_index_valid[position]\n    actual_factors_1.append(\n        valid * (\n            challenge_1 - (locator + alpha_1 * block + beta_1 * bucket)\n        ) + (1 - valid)\n    )\n    actual_factors_2.append(\n        valid * (\n            challenge_2 - (locator + alpha_2 * block + beta_2 * bucket)\n        ) + (1 - valid)\n    )\n\nrelation_ok = (\n    (product_tree(expected_factors_1) == product_tree(actual_factors_1))\n    * (product_tree(expected_factors_2) == product_tree(actual_factors_2))\n)\n\ncommitted_db_locators = sint.Array(N_RECORDS)\ncommitted_index_locators = sint.Array(N_INDEX_SLOTS)\ncommitted_index_valid = sint.Array(N_INDEX_SLOTS)\n\nfor row in range(N_RECORDS):\n    committed_db_locators[row] = relation_ok.if_else(\n        new_db_locators[row], old_db_locators[row]\n    )\n\nfor position in range(N_INDEX_SLOTS):\n    committed_index_locators[position] = relation_ok.if_else(\n        new_index_locators[position], old_index_locators[position]\n    )\n    committed_index_valid[position] = relation_ok.if_else(\n        new_index_valid[position], index_valid[position]\n    )\n\ncommitted_db_locators.write_to_file()\ncommitted_index_locators.write_to_file()\ncommitted_index_valid.write_to_file()\n\nopened_relation_ok = relation_ok.reveal()\nstop_timer(5)\n\nprint_ln(\n    "SLE_MPC_RESULT mode=oram_reindex epoch=%s relation_ok=%s",\n    NEW_EPOCH,\n    opened_relation_ok,\n)\nprint_ln(\n    "SLE_MPC_PRIVATE key=not_opened locators=not_opened "\n    "access_pattern=oram_hidden row_references=not_opened"\n)\n', encoding='utf-8')
print('Wrote:', target)


In [ ]:
# This cell reconstructs one reviewed project file inside Colab.
from pathlib import Path
target = Path('/content/sle_sota_project/scripts/generate_inputs.py')
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('#!/usr/bin/env python3\n"""Generate reproducible MP-SPDZ inputs for the SLE MPC programs.\n\nThe default path creates a small synthetic gallery/probe pair so the\ncryptographic layer can be tested without downloading a face dataset. An NPZ\nexport from the biometric notebook can be supplied later; the MPC programs see\nonly binary roots, never face images or real-valued FaceNet embeddings.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\n\n\nSEED = 20260720\n\n\n@dataclass\nclass ProtectedInputs:\n    gallery: np.ndarray\n    labels: np.ndarray\n    tie_tags: np.ndarray\n    probe: np.ndarray\n    query_label: int\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--mode", choices=("full_scan", "oram", "transition"), required=True)\n    parser.add_argument("--output-dir", type=Path, required=True)\n    parser.add_argument("--n-records", type=int, default=16)\n    parser.add_argument("--root-bits", type=int, default=64)\n    parser.add_argument("--block-bits", type=int, default=4)\n    parser.add_argument("--bucket-capacity", type=int, default=8)\n    parser.add_argument("--query-row", type=int, default=0)\n    parser.add_argument("--flip-probability", type=float, default=0.05)\n    parser.add_argument("--npz", type=Path)\n    return parser.parse_args()\n\n\ndef load_or_generate(args: argparse.Namespace) -> ProtectedInputs:\n    rng = np.random.default_rng(SEED)\n\n    if args.npz is not None:\n        archive = np.load(args.npz)\n        gallery = np.asarray(archive["gallery_roots"], dtype=np.int64)\n        labels = np.asarray(\n            archive.get("labels", np.arange(len(gallery))), dtype=np.int64\n        )\n        probes = np.asarray(archive["probe_roots"], dtype=np.int64)\n        if gallery.ndim != 2 or probes.ndim != 2:\n            raise ValueError("gallery_roots and probe_roots must be two-dimensional")\n        if gallery.shape != probes.shape:\n            raise ValueError("gallery_roots and probe_roots must have equal shape")\n        if len(gallery) < args.n_records or gallery.shape[1] < args.root_bits:\n            raise ValueError("NPZ does not contain enough records/root bits")\n        gallery = gallery[: args.n_records, : args.root_bits]\n        labels = labels[: args.n_records]\n        probe = probes[args.query_row, : args.root_bits]\n    else:\n        gallery = rng.integers(\n            0, 2, size=(args.n_records, args.root_bits), dtype=np.int64\n        )\n        labels = np.arange(1000, 1000 + args.n_records, dtype=np.int64)\n\n        # Guarantee that the first flattened index slot is real. This is used\n        # only by the explicit fault-injection smoke tests.\n        gallery[0, : args.block_bits] = 0\n\n        probe = gallery[args.query_row].copy()\n        flips = rng.random(args.root_bits) < args.flip_probability\n        probe[flips] ^= 1\n\n    if not np.isin(gallery, (0, 1)).all() or not np.isin(probe, (0, 1)).all():\n        raise ValueError("MPC roots must be binary")\n\n    tie_tags = rng.permutation(np.arange(1, args.n_records + 1, dtype=np.int64))\n    return ProtectedInputs(\n        gallery=gallery,\n        labels=labels,\n        tie_tags=tie_tags,\n        probe=probe,\n        query_label=int(labels[args.query_row]),\n    )\n\n\ndef block_value(root: np.ndarray, block: int, block_bits: int) -> int:\n    start = block * block_bits\n    value = 0\n    for bit, item in enumerate(root[start : start + block_bits]):\n        value |= int(item) << bit\n    return value\n\n\ndef build_bucket_rows(\n    gallery: np.ndarray,\n    block_bits: int,\n    capacity: int,\n) -> tuple[np.ndarray, np.ndarray]:\n    n_records, root_bits = gallery.shape\n    if root_bits % block_bits:\n        raise ValueError("root_bits must be divisible by block_bits")\n\n    n_blocks = root_bits // block_bits\n    bucket_values = 2**block_bits\n    n_buckets = n_blocks * bucket_values\n    dummy_row = n_records\n    rows = np.full((n_buckets, capacity), dummy_row, dtype=np.int64)\n    valid = np.zeros((n_buckets, capacity), dtype=np.int64)\n    occupancy = np.zeros(n_buckets, dtype=np.int64)\n\n    for row in range(n_records):\n        for block in range(n_blocks):\n            value = block_value(gallery[row], block, block_bits)\n            address = block * bucket_values + value\n            slot = int(occupancy[address])\n            if slot >= capacity:\n                raise ValueError(\n                    f"bucket {address} requires more than {capacity} slots; "\n                    "increase --bucket-capacity"\n                )\n            rows[address, slot] = row\n            valid[address, slot] = 1\n            occupancy[address] += 1\n\n    return rows, valid\n\n\ndef flatten(*arrays: np.ndarray | list[int] | tuple[int, ...]) -> list[int]:\n    output: list[int] = []\n    for array in arrays:\n        output.extend(np.asarray(array, dtype=np.int64).reshape(-1).tolist())\n    return output\n\n\ndef write_inputs(output_dir: Path, values_by_player: dict[int, list[int]]) -> None:\n    output_dir.mkdir(parents=True, exist_ok=True)\n    for player in range(5):\n        values = values_by_player.get(player, [])\n        path = output_dir / f"Input-P{player}-0"\n        path.write_text(" ".join(str(int(value)) for value in values) + "\\n")\n\n\ndef full_scan_inputs(data: ProtectedInputs) -> dict[int, list[int]]:\n    return {\n        0: flatten(data.labels, data.tie_tags, data.gallery),\n        1: flatten(data.probe),\n    }\n\n\ndef oram_inputs(\n    data: ProtectedInputs,\n    block_bits: int,\n    capacity: int,\n) -> tuple[dict[int, list[int]], np.ndarray, np.ndarray]:\n    rows, valid_slots = build_bucket_rows(data.gallery, block_bits, capacity)\n    n_records, root_bits = data.gallery.shape\n\n    labels = np.concatenate((data.labels, [-1]))\n    valid_records = np.concatenate((np.ones(n_records, dtype=np.int64), [0]))\n    tie_tags = np.concatenate((data.tie_tags, [2**31 - 1]))\n    gallery = np.vstack((data.gallery, np.zeros(root_bits, dtype=np.int64)))\n\n    return (\n        {\n            0: flatten(labels, valid_records, tie_tags, gallery, rows),\n            1: flatten(data.probe),\n        },\n        rows,\n        valid_slots,\n    )\n\n\ndef transition_inputs(\n    data: ProtectedInputs,\n    block_bits: int,\n    capacity: int,\n) -> tuple[dict[int, list[int]], np.ndarray, np.ndarray]:\n    rows, valid_slots = build_bucket_rows(data.gallery, block_bits, capacity)\n    rng = np.random.default_rng(SEED + 1)\n    # Generate only the required unique locators. Building\n    # np.arange(1, 2**31) would allocate roughly 16 GiB before sampling and\n    # can make a Colab runtime terminate with exit code 137.\n    unique_locators: set[int] = set()\n    while len(unique_locators) < len(data.gallery):\n        unique_locators.add(int(rng.integers(1, 2**31)))\n    old_db_locators = np.asarray(sorted(unique_locators), dtype=np.int64)\n    dummy_locators = rng.integers(\n        1, 2**31, size=rows.shape, dtype=np.int64\n    )\n    # The pointer-join baseline requires every dummy slot to match zero real\n    # database locators. Make that invariant deterministic instead of relying\n    # on the already tiny probability of a random 31-bit collision.\n    collides = np.isin(dummy_locators, old_db_locators)\n    while collides.any():\n        dummy_locators[collides] = rng.integers(\n            1, 2**31, size=int(collides.sum()), dtype=np.int64\n        )\n        collides = np.isin(dummy_locators, old_db_locators)\n    old_index_locators = dummy_locators.copy()\n    for address in range(rows.shape[0]):\n        for slot in range(rows.shape[1]):\n            if valid_slots[address, slot]:\n                old_index_locators[address, slot] = old_db_locators[\n                    rows[address, slot]\n                ]\n\n    # Each party contributes one independent key component. The MPC program\n    # adds them while shared, so no party observes the complete epoch key.\n    contributions = rng.integers(1, 2**30, size=5, dtype=np.int64)\n    player_values = {\n        0: flatten(\n            old_db_locators,\n            data.gallery,\n            old_index_locators,\n            valid_slots,\n            [contributions[0]],\n            # Persistent secret row references are consumed only by the\n            # ORAM-reindexing baseline.  Existing SLE/join/sort programs stop\n            # after the key contribution and safely ignore these trailing\n            # test inputs.\n            rows,\n        )\n    }\n    for player in range(1, 5):\n        player_values[player] = [int(contributions[player])]\n    return player_values, rows, valid_slots\n\n\ndef main() -> None:\n    args = parse_args()\n    if args.n_records < 2:\n        raise ValueError("--n-records must be at least two")\n    if args.root_bits % args.block_bits:\n        raise ValueError("--root-bits must be divisible by --block-bits")\n\n    data = load_or_generate(args)\n    metadata: dict[str, object] = {\n        "mode": args.mode,\n        "n_records": args.n_records,\n        "root_bits": args.root_bits,\n        "block_bits": args.block_bits,\n        "bucket_capacity": args.bucket_capacity,\n        "query_row": args.query_row,\n        "expected_identity": data.query_label,\n        "plain_query_distance": int(\n            np.count_nonzero(data.gallery[args.query_row] != data.probe)\n        ),\n    }\n\n    if args.mode == "full_scan":\n        player_values = full_scan_inputs(data)\n    elif args.mode == "oram":\n        player_values, rows, valid = oram_inputs(\n            data, args.block_bits, args.bucket_capacity\n        )\n        metadata.update(\n            padded_slots=int(rows.size),\n            real_slots=int(valid.sum()),\n            maximum_occupancy=int(valid.sum(axis=1).max()),\n        )\n    else:\n        player_values, rows, valid = transition_inputs(\n            data, args.block_bits, args.bucket_capacity\n        )\n        metadata.update(\n            padded_slots=int(rows.size),\n            real_relation_entries=int(valid.sum()),\n            maximum_occupancy=int(valid.sum(axis=1).max()),\n        )\n\n    write_inputs(args.output_dir, player_values)\n    (args.output_dir / "sle_input_metadata.json").write_text(\n        json.dumps(metadata, indent=2), encoding="utf-8"\n    )\n    print(json.dumps(metadata, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('Wrote:', target)


In [ ]:
# This cell reconstructs one reviewed project file inside Colab.
from pathlib import Path
target = Path('/content/sle_sota_project/scripts/verify_outputs.py')
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('#!/usr/bin/env python3\n"""Check the intentionally small public output of an MP-SPDZ SLE run."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport re\nfrom pathlib import Path\n\n\nRESULT_RE = re.compile(\n    r"SLE_MPC_RESULT mode=(?P<mode>\\w+).*?"\n    r"(?:accepted=(?P<accepted>[01]).*?identity=(?P<identity>-?\\d+)|"\n    r"relation_ok=(?P<relation_ok>[01]))"\n)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--log", type=Path, required=True)\n    parser.add_argument("--metadata", type=Path)\n    parser.add_argument("--expect-relation", type=int, choices=(0, 1))\n    args = parser.parse_args()\n\n    text = args.log.read_text(encoding="utf-8", errors="replace")\n    match = RESULT_RE.search(text)\n    if match is None:\n        raise SystemExit("No SLE_MPC_RESULT line found")\n\n    result = match.groupdict()\n    if result["mode"] in ("full_scan", "oram"):\n        if args.metadata is None:\n            raise SystemExit("--metadata is required for identification modes")\n        metadata = json.loads(args.metadata.read_text(encoding="utf-8"))\n        expected = int(metadata["expected_identity"])\n        observed = int(result["identity"])\n        accepted = int(result["accepted"])\n        if accepted != 1 or observed != expected:\n            raise SystemExit(\n                f"Identification failed: expected {expected}, observed {observed}, "\n                f"accepted={accepted}"\n            )\n        print(f"PASS: {result[\'mode\']} opened only the expected identity {expected}")\n    else:\n        expected = args.expect_relation\n        observed = int(result["relation_ok"])\n        if expected is None:\n            raise SystemExit("--expect-relation is required for transition mode")\n        if observed != expected:\n            raise SystemExit(\n                f"Transition check failed: expected {expected}, observed {observed}"\n            )\n        print(f"PASS: transition relation_ok={observed}")\n\n\nif __name__ == "__main__":\n    main()\n\n', encoding='utf-8')
print('Wrote:', target)


In [ ]:
# This cell reconstructs one reviewed project file inside Colab.
from pathlib import Path
target = Path('/content/sle_sota_project/SLE_FORMAL_PROTOCOL.md')
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('# SLE: formal protocol and experimental claims\n\n## 1. Scope\n\nSynchronized Locator Evolution (SLE) is the epoch-transition component of a\nprivate indexed 1:N biometric identification system. SLE is not a new MPC,\nORAM, PRF, secret-sharing, or biometric-hashing primitive. Its proposed\ncontribution is the way independently shared copies of one semantic locator\nare evolved and verified across a protected biometric database and its padded\ncandidate index without constructing an explicit old-to-new pointer map.\n\n## 2. Parties and adversary\n\n- `P0,...,P4` are five MPC computing parties.\n- The current executable backend is Shamir sharing with threshold `t=2`.\n- The implemented security model is passive/semi-honest.\n- At least one key contribution must remain unknown to the adversary.\n- Enrollment roots, probes, locators, distances, candidates, and epoch keys\n  remain shared. Only Top-1/reject or transition accept/reject is opened.\n- Active fault injection tests protocol invariants; it does not turn passive\n  Shamir into an actively secure MPC protocol.\n\n## 3. State\n\nAt epoch `e`, the protected state contains:\n\n```text\nD_e[i] = ([ell_i^e], [b_i], [label_i])\nH_e[m,v,s] = ([ell^e], [valid])\n```\n\nHere `b_i` is a binary biometric root, `ell_i^e` is its secret semantic\nlocator, `m` is a public root block, `v` is a public block value, and `s` is a\nslot in a fixed-capacity padded bucket. Database and index occurrences of the\nsame semantic locator are independent secret sharings.\n\n## 4. Identification\n\nGiven shared probe root `[q]`:\n\n1. Derive `L=d/block_bits` secret bucket addresses from `[q]`.\n2. Read fixed-capacity candidate lists with MPC-ORAM.\n3. Read candidate roots with MPC-ORAM.\n4. Compute exact Hamming distances inside MPC.\n5. Apply deterministic secure ArgMin and the public acceptance threshold.\n6. Open only identity/reject.\n\nThe executable comparison is:\n\n```text\nfull scan: O(N*d) secure matching, no secret index\nindexed query: ORAM retrieval + O(C*d) secure matching\n```\n\nThe indexed design is considered faster only after measured runtime and\ncommunication beat full scan under the same MPC backend and hardware.\n\n## 5. SLE transition\n\n### Inputs\n\n- Old database locators `[ell_i^e]`.\n- Old padded-index locator copies and valid bits.\n- Binary roots `[b_i]` used only to derive the expected bucket relation.\n- One secret key contribution `[k_j]` from each party.\n\n### Algorithm\n\n```text\nK_(e+1) <- sum_j [k_j]\n\nfor every database locator copy:\n    [ell_i^(e+1)] <- PRP_MPC(K_(e+1), e+1, [ell_i^e])\n\nfor every independently shared index locator copy:\n    [ell^(e+1)] <- PRP_MPC(K_(e+1), e+1, [ell^e])\n\nexpected <- {(ell_i^(e+1), m, h_m(b_i)) for every i,m}\nactual   <- {(ell^(e+1), m, v) for every valid index slot (m,v,s)}\n\nok <- MultisetFingerprint(expected) == MultisetFingerprint(actual)\n\nif ok:\n    atomically commit the new shared state\nelse:\n    retain the old shared state\n\nopen only ok\n```\n\nSLE applies the same secret transformation independently to semantically\nequal locator copies. It never asks where the database row moved and never\nproduces an old-row/new-row pointer relation.\n\n## 6. Integrity check\n\nFor public challenge `r` and tuple encoding `E`, each multiset is represented\nby:\n\n```text\nPhi(S,r) = product_(x in S) (r - E(x)).\n```\n\nTwo independently sampled challenges are used. Assuming non-colliding tuple\nencoding and challenges sampled after candidate values are fixed, an unequal\nrelation is accepted only with the polynomial identity-test error. The exact\nbound and commitment order must be proved for the final field and malicious\nbackend; the current implementation is an empirical invariant check.\n\n## 7. Executable transition baseline\n\n`sle_pointer_join_transition.mpc` represents generic secret pointer\ntranslation. It evolves `N` database locators and, for each of `M` index\nentries, performs secret equality against all old database locators:\n\n```text\n[new_index_j] = sum_i [old_index_j == old_db_i] * [new_db_i].\n```\n\nIt reveals no map but requires `O(N*M)` secure equality work. Both SLE and the\nbaseline use the same key construction, relation check, atomic commit,\nbackend, and opened output. This isolates the proposed pointerless transition.\n\n## 8. Falsifiable hypotheses\n\nThe method claim survives only if all mandatory hypotheses hold.\n\n| ID | Hypothesis | Required evidence |\n|---|---|---|\n| H1 | SLE preserves identification semantics. | Equal Top-1/reject decisions before and after accepted transitions. |\n| H2 | SLE detects invalid epoch states. | Omission, duplication, locator corruption, and wrong-bucket tests are rejected. |\n| H3 | SLE reduces transition cost. | Lower runtime, communication, rounds, or secure comparisons than pointer join at equal leakage. |\n| H4 | Evolving state reduces cross-epoch linkage. | Graph/linking attack AUC approaches 0.5 without unacceptable identification loss. |\n| H5 | Indexed search scales better than full scan. | A measured crossover as `N` grows under the same MPC protocol. |\n\nFailure of H3 removes the efficiency claim but may leave a lifecycle/security\ncontribution. Failure of H2 or H4 invalidates the main SLE security objective.\n\n## 9. Benchmark protocol\n\nFor every configuration, report:\n\n- `N`, root bits, block bits, candidate capacity, real/padded index entries;\n- compile time, secure timer, total runtime, global communication, and rounds;\n- accepted identity or transition bit;\n- protocol, party count, threshold, machine, and network setting;\n- mean, standard deviation, and confidence interval over at least five runs.\n\nThe minimum scaling grid is `N in {8,16,32,64,128}` with fixed 32-bit roots\nfor cryptographic scaling, followed by realistic 512/1024-bit roots on the\nlargest configuration that fits. Synthetic roots test cryptographic wiring;\nFaceNet/CelebA roots test biometric behavior. These result classes must not be\nmixed.\n\n## 10. Literature-facing comparison boundary\n\n- IDFace and PFIP are encrypted biometric-identification references. Their\n  published latency cannot be treated as a direct speed baseline unless the\n  dataset, hardware, output, threat model, and accuracy protocol are aligned.\n- GigaDORAM is infrastructure for secret indexed access, not an SLE competitor.\n- Use-Once KVS, secret shuffle, proactive sharing, and distributed PRFs are\n  building blocks or adjacent infrastructure; none should be claimed as new.\n- The direct executable algorithmic baseline is generic secret pointer\n  translation. Full scan is the direct query baseline.\n\n## 11. Current claim status\n\nThe smoke experiment establishes functional feasibility only: SLE can execute\ninside five-party MPC and the tested relation faults are detected. It does not\nyet establish active security, biometric accuracy, cross-epoch unlinkability,\nor scalable advantage. Those claims become available only after H1-H5 are\nmeasured and the formal proof assumptions are reviewed.\n', encoding='utf-8')
print('Wrote:', target)


In [ ]:
# This cell reconstructs one reviewed project file inside Colab.
from pathlib import Path
target = Path('/content/sle_sota_project/SLE_SOTA_NUMERICAL_COMPARISON_RU.md')
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text('# Численное сравнение SLE с ближайшими SOTA-подходами\n\n## Что теперь измеряется\n\n`sle_sota_comparison_colab.ipynb` выполняет одну и ту же операцию перехода\nэпохи четырьмя способами на одинаковых секретных входах и под одним\nпятипартийным Shamir backend в MP-SPDZ:\n\n1. **SLE**: синхронно применяет свежий keyed PRP ко всем копиям locator.\n2. **ORAM reindexing adaptation**: хранит persistent secret row reference и\n   получает новый locator через скрытый `OptimalORAM` lookup.\n3. **Oblivious join/sort adaptation**: соединяет старые и новые locator через\n   oblivious sort/merge по скрытому ключу.\n4. **Quadratic pointer join**: проверяет каждую пару index/database; это\n   контрольный generic MPC baseline, а не SOTA.\n\nВсе варианты включают одинаковую проверку скрытого отношения, atomic commit и\nrollback. Публично раскрывается только `relation_ok`.\n\n## Какие цифры получаются\n\nNotebook сохраняет:\n\n- `sle_paper_ready_numerical_comparison.csv`: готовая итоговая таблица;\n- `sle_sota_direct_ci95.csv`: средние значения и bootstrap 95% CI;\n- `sle_sota_direct_raw.csv`: все повторы;\n- `sle_sota_interpretation.txt`: автоматически составленные предложения с\n  абсолютными значениями и коэффициентами.\n\nДля каждого `N` итоговая строка содержит:\n\n| Поле | Смысл |\n|---|---|\n| `sle_seconds` | runtime SLE |\n| `baseline_seconds` | runtime сравниваемого подхода |\n| `speedup_x` | во сколько раз SLE быстрее |\n| `sle_global_mb` | общая коммуникация SLE |\n| `baseline_global_mb` | коммуникация baseline |\n| `communication_reduction_x` | во сколько раз SLE передает меньше данных |\n\n## Уже полученные quick-run цифры\n\nПредыдущий одинаковый MP-SPDZ запуск при одном повторе показал:\n\n| N | Сравнение | Runtime SLE / baseline | Ускорение | Communication SLE / baseline | Снижение communication |\n|---:|---|---:|---:|---:|---:|\n| 8 | quadratic pointer join | 1.146 / 8.019 s | 7.00x | 9.61 / 32.82 MB | 3.42x |\n| 16 | quadratic pointer join | 1.123 / 7.004 s | 6.24x | 9.82 / 58.15 MB | 5.92x |\n| 8 | oblivious join/sort adaptation | 1.146 / 45.406 s | 39.62x | 9.61 / 430.37 MB | 44.81x |\n| 16 | oblivious join/sort adaptation | 1.123 / 45.823 s | 40.81x | 9.82 / 431.60 MB | 43.94x |\n\nЭто предварительные цифры: один повтор и малые `N`. ORAM-reindexing добавлен в\nновый notebook и должен быть измерен тем же запуском.\n\n## Допустимая формулировка для статьи\n\nПосле paper-mode запуска допустима формулировка следующего вида:\n\n> Under an identical five-party MP-SPDZ configuration, SLE completed the\n> verified coupled epoch transition in **A s**, compared with **B s** for the\n> ORAM-based reindexing adaptation and **C s** for the oblivious join/sort\n> adaptation, yielding **B/A x** and **C/A x** speedups. SLE communicated\n> **D MB**, corresponding to **E x** and **F x** reductions, respectively.\n> All methods passed the same hidden-relation verification.\n\nТочные `A-F` автоматически появятся в `sle_sota_interpretation.txt`.\n\n## Граница claim\n\nКорректно: **SLE численно превосходит измеренные SOTA-derived ORAM и\noblivious-join/sort adaptations на задаче verified coupled epoch transition**.\n\nНекорректно: **SLE быстрее GigaDORAM, PRAC или MRR вообще**. Эти системы\nрешают другие операции, используют другие party counts, threat models,\nhardware и outputs. Их опубликованные цифры находятся в отдельной таблице\n`sle_published_sota_context.csv` и не используются для искусственного ratio.\n\nОсновные источники контекста:\n\n- Mohassel, Rindal, Rosulek, secure database joins, CCS 2020.\n- Chase, Ghosh, Poburinnaya, secret-shared shuffle, ASIACRYPT 2020.\n- Song et al., malicious secret-shared shuffle, NDSS 2024.\n- GigaDORAM, USENIX Security 2023.\n- PRAC/Duoram, PoPETs 2024.\n', encoding='utf-8')
print('Wrote:', target)


In [ ]:
# CELL 9. Install embedded sources into MP-SPDZ
project = Path('/content/sle_sota_project')
mpspdz = Path('/content/MP-SPDZ')
runs_dir = project / 'runs'
runs_dir.mkdir(parents=True, exist_ok=True)

program_names = [
    'sle_transition_optimized.mpc',
    'sle_sort_merge_transition.mpc',
    'sle_pointer_join_transition.mpc',
    'sle_oram_reindex_transition.mpc',
]
for name in program_names:
    shutil.copy2(
        project / 'Programs/Source' / name,
        mpspdz / 'Programs/Source' / name,
    )

inputs = mpspdz / 'Player-Data'
env = os.environ.copy()
env.update(PLAYERS=str(PLAYERS), THRESHOLD=str(THRESHOLD))
print('Installed protocol sources:', program_names)


## Public padding capacity

The candidate index uses fixed-size buckets. Capacity is computed from
the same gallery used for the run and then treated as a public system
parameter. The table also shows `M`, because the transition operates on
every padded slot, not only on the `N` real identities.


In [ ]:
# CELL 10. Build benchmark configurations
SEED = 20260720

def gallery_for_capacity(n, root_bits, npz_path=None):
    if npz_path:
        archive = np.load(npz_path)
        gallery = np.asarray(archive['gallery_roots'], dtype=np.int64)
        if len(gallery) < n or gallery.shape[1] < root_bits:
            raise ValueError(f'NPZ is too small for N={n}, d={root_bits}')
        return gallery[:n, :root_bits]

    rng = np.random.default_rng(SEED)
    gallery = rng.integers(0, 2, size=(n, root_bits), dtype=np.int64)
    gallery[0, :BLOCK_BITS] = 0
    return gallery

def required_capacity(gallery, block_bits):
    n, d = gallery.shape
    values = 2 ** block_bits
    occupancy = np.zeros((d // block_bits, values), dtype=np.int64)
    for root in gallery:
        for block in range(d // block_bits):
            value = sum(
                int(root[block * block_bits + bit]) << bit
                for bit in range(block_bits)
            )
            occupancy[block, value] += 1
    return int(occupancy.max())

CONFIGS = []
for n in ACTIVE_N:
    gallery = gallery_for_capacity(n, ROOT_BITS, REAL_ROOTS_NPZ)
    capacity = required_capacity(gallery, BLOCK_BITS)
    padded_slots = (
        (ROOT_BITS // BLOCK_BITS)
        * (2 ** BLOCK_BITS)
        * capacity
    )
    CONFIGS.append(dict(
        n=n,
        root_bits=ROOT_BITS,
        block_bits=BLOCK_BITS,
        capacity=capacity,
        padded_slots=padded_slots,
        combined_rows=n + padded_slots,
    ))

configs_df = pd.DataFrame(CONFIGS)
display(configs_df)


## Measurement helpers

`secure_seconds` is the timer placed around the transition logic.
`total_seconds` is MP-SPDZ runtime including protocol preprocessing
generated during execution. `global_mb` is aggregate communication.
Compiler VM rounds and triple counts are also retained.


In [ ]:
# CELL 11. Parse compile and runtime logs
def last_float(pattern, text):
    values = re.findall(pattern, text, flags=re.MULTILINE)
    return float(values[-1]) if values else np.nan

def last_int(pattern, text):
    values = re.findall(pattern, text, flags=re.MULTILINE)
    return int(values[-1]) if values else np.nan

def parse_measurement(
    mode, cfg, repeat, compile_text, run_text,
    compile_seconds, wall_seconds,
):
    timer_values = [
        float(value)
        for value in re.findall(r'timer \d+ at ([0-9.eE+-]+)', run_text)
    ]
    result_lines = re.findall(r'SLE_MPC_RESULT[^\n]+', run_text)
    return dict(
        mode=mode,
        n=cfg['n'],
        root_bits=cfg['root_bits'],
        block_bits=cfg['block_bits'],
        capacity=cfg['capacity'],
        padded_slots=cfg['padded_slots'],
        combined_rows=cfg['combined_rows'],
        repeat=repeat,
        passed=bool(result_lines and 'relation_ok=1' in result_lines[-1]),
        result=result_lines[-1] if result_lines else None,
        compile_seconds=compile_seconds,
        wall_seconds=wall_seconds,
        secure_seconds=max(timer_values) if timer_values else np.nan,
        total_seconds=last_float(r'Time = ([0-9.eE+-]+)', run_text),
        party0_mb=last_float(r'Data sent = ([0-9.eE+-]+) MB', run_text),
        global_mb=last_float(r'Global data sent = ([0-9.eE+-]+) MB', run_text),
        runtime_rounds=last_int(r'Data sent = [^\n]*~(\d+) rounds', run_text),
        compiler_vm_rounds=last_int(r'(\d+) virtual machine rounds', compile_text),
        integer_triples=last_int(r'(\d+) integer triples', compile_text),
    )

def npz_flag():
    return f' --npz {REAL_ROOTS_NPZ}' if REAL_ROOTS_NPZ else ''

measurements = []


## Same-backend transition benchmark

Every variant receives identical secret inputs. Compilation happens
once per `(method,N)` and runtime is repeated `REPEATS` times. A row is
accepted only after the verifier confirms `relation_ok=1`.


In [ ]:
# CELL 12. Compile and run all same-backend transition methods
for cfg in CONFIGS:
    n = cfg['n']
    d = cfg['root_bits']
    b = cfg['block_bits']
    c = cfg['capacity']

    run(
        f'python3 {project}/scripts/generate_inputs.py '
        f'--mode transition --output-dir {inputs} '
        f'--n-records {n} --root-bits {d} '
        f'--block-bits {b} --bucket-capacity {c}'
        + npz_flag()
    )

    variants = []
    if RUN_SLE:
        variants.append((
            'sle',
            'sle_transition_optimized',
            f'{n} {d} {b} {c} {EPOCH} 0',
        ))
    if RUN_SORT_MERGE and n <= SORT_MERGE_MAX_N:
        variants.append((
            'sort_merge',
            'sle_sort_merge_transition',
            f'{n} {d} {b} {c} {EPOCH}',
        ))
    if RUN_ORAM_REINDEX and n <= ORAM_REINDEX_MAX_N:
        variants.append((
            'oram_reindex',
            'sle_oram_reindex_transition',
            f'{n} {d} {b} {c} {EPOCH}',
        ))
    if RUN_POINTER_JOIN and n <= POINTER_JOIN_MAX_N:
        variants.append((
            'pointer_join',
            'sle_pointer_join_transition',
            f'{n} {d} {b} {c} {EPOCH}',
        ))

    for mode, source_name, arguments in variants:
        compile_log = runs_dir / f'{mode}_n{n}_compile.log'
        compile_text, compile_seconds = run(
            f'python3 compile.py -M {source_name} {arguments}',
            cwd=mpspdz,
            log_path=compile_log,
        )
        program_name = source_name + '-' + arguments.replace(' ', '-')

        for repeat in range(REPEATS):
            runtime_log = runs_dir / f'{mode}_n{n}_r{repeat}_run.log'
            run_text, wall_seconds = run(
                f'Scripts/shamir.sh {program_name}',
                cwd=mpspdz,
                env=env,
                log_path=runtime_log,
            )
            run(
                f'python3 {project}/scripts/verify_outputs.py '
                f'--log {runtime_log} --expect-relation 1'
            )
            measurements.append(parse_measurement(
                mode,
                cfg,
                repeat,
                compile_text,
                run_text,
                compile_seconds,
                wall_seconds,
            ))

raw = pd.DataFrame(measurements)
if raw.empty:
    raise RuntimeError('No benchmark variant was enabled')
if not raw['passed'].all():
    raise RuntimeError('At least one transition failed relation verification')
display(raw)


## Direct results

Ratios are computed only for methods actually measured at the same
`N`. `runtime_vs_sle > 1` means SLE was faster by that factor;
`communication_vs_sle > 1` means SLE sent less data by that factor.
Missing sizes are not extrapolated.


In [ ]:
# CELL 13. Aggregate results, confidence intervals, and direct ratios
metrics = [
    'secure_seconds', 'total_seconds', 'global_mb',
    'runtime_rounds', 'compiler_vm_rounds', 'integer_triples',
]
direct_summary = (
    raw.groupby(['mode', 'n'], as_index=False)[metrics]
    .mean(numeric_only=True)
)

def mean_ci95(values, seed=20260803):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return float(values.mean()) if len(values) else np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    samples = rng.choice(values, size=(10000, len(values)), replace=True)
    means = samples.mean(axis=1)
    return float(values.mean()), *np.quantile(means, [0.025, 0.975])

ci_rows = []
for (mode, n), group in raw.groupby(['mode', 'n']):
    row = dict(mode=mode, n=n, repeats=len(group))
    for metric in ('total_seconds', 'global_mb', 'runtime_rounds'):
        mean, low, high = mean_ci95(group[metric])
        row[f'{metric}_mean'] = mean
        row[f'{metric}_ci_low'] = low
        row[f'{metric}_ci_high'] = high
    ci_rows.append(row)
direct_ci = pd.DataFrame(ci_rows)

ratio_rows = []
for n in sorted(raw['n'].unique()):
    at_n = direct_summary[direct_summary['n'] == n]
    sle_rows = at_n[at_n['mode'] == 'sle']
    if sle_rows.empty:
        continue
    sle_row = sle_rows.iloc[0]
    for _, row in at_n.iterrows():
        ratio_rows.append(dict(
            n=n,
            method=row['mode'],
            total_seconds=row['total_seconds'],
            global_mb=row['global_mb'],
            runtime_rounds=row['runtime_rounds'],
            runtime_vs_sle=(
                row['total_seconds'] / sle_row['total_seconds']
            ),
            communication_vs_sle=(
                row['global_mb'] / sle_row['global_mb']
            ),
        ))

direct_ratios = pd.DataFrame(ratio_rows)
display(direct_summary)
display(direct_ci)
display(direct_ratios)

raw.to_csv(project / 'sle_sota_direct_raw.csv', index=False)
direct_summary.to_csv(project / 'sle_sota_direct_summary.csv', index=False)
direct_ci.to_csv(project / 'sle_sota_direct_ci95.csv', index=False)
direct_ratios.to_csv(project / 'sle_sota_direct_ratios.csv', index=False)
configs_df.to_csv(project / 'sle_sota_configurations.csv', index=False)


In [ ]:
# CELL 14. Scientific-style scaling figure
method_labels = {
    'sle': 'SLE',
    'sort_merge': 'Oblivious join/sort adaptation',
    'oram_reindex': 'ORAM reindexing adaptation',
    'pointer_join': 'Quadratic pointer join',
}
method_styles = {
    'sle': dict(color='#111111', marker='o'),
    'sort_merge': dict(color='#0072B2', marker='s'),
    'oram_reindex': dict(color='#009E73', marker='D'),
    'pointer_join': dict(color='#D55E00', marker='^'),
}

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.2))
plot_specs = [
    ('total_seconds', 'Total runtime (s)'),
    ('global_mb', 'Global communication (MB)'),
    ('runtime_rounds', 'Runtime rounds'),
]

for mode, group in direct_summary.groupby('mode'):
    group = group.sort_values('n')
    style = method_styles[mode]
    for axis, (column, ylabel) in zip(axes, plot_specs):
        axis.plot(
            group['n'], group[column],
            linewidth=1.8, markersize=6,
            label=method_labels[mode], **style,
        )
        axis.set_xlabel('Enrolled identities, N')
        axis.set_ylabel(ylabel)
        axis.grid(True, linestyle=':', linewidth=0.8, alpha=0.55)

axes[0].legend(frameon=False)
fig.suptitle('Epoch-transition cost under identical five-party Shamir MPC')
fig.tight_layout()
figure_path = project / 'sle_sota_same_backend_comparison.png'
fig.savefig(figure_path, dpi=240, bbox_inches='tight')
plt.show()


## Exact named artifact: PRAC/Duoram

This section checks out the pinned official PRAC `popets-repro`
artifact and executes its documented three-party native Duoram
workload. It is an **exact artifact run**, but not the same operation
as SLE's verified coupled epoch transition. Its timing is therefore
reported as native SOTA context and is never divided by SLE runtime.

The first clean Colab build normally takes a few minutes. Re-running
the cell in the same session reuses the compiled binary.


In [ ]:
# CELL 15. Build and execute the pinned official PRAC artifact
import traceback

PRAC_URL = 'https://git-crysp.uwaterloo.ca/iang/prac.git'
prac_dir = Path('/content/prac_sota_artifact')
prac_logs = runs_dir / 'prac_exact'
prac_logs.mkdir(parents=True, exist_ok=True)

def run_bounded(command, cwd=None, timeout=900, log_path=None):
    print('$', command)
    started = time.perf_counter()
    completed = subprocess.run(
        command,
        cwd=cwd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
    )
    elapsed = time.perf_counter() - started
    output = completed.stdout or ''
    print(output[-6000:])
    print(f'elapsed={elapsed:.2f}s returncode={completed.returncode}')
    if log_path is not None:
        Path(log_path).write_text(output, encoding='utf-8')
    if completed.returncode:
        raise RuntimeError(
            f'command failed ({completed.returncode}): {command}'
        )
    return output, elapsed

def run_party_group(commands, cwd, timeout=900):
    processes = []
    handles = []
    log_paths = []
    started = time.perf_counter()
    try:
        for party, command in enumerate(commands):
            log_path = prac_logs / f'party{party}.log'
            handle = open(log_path, 'w', encoding='utf-8')
            process = subprocess.Popen(
                command,
                cwd=cwd,
                shell=True,
                text=True,
                stdout=handle,
                stderr=subprocess.STDOUT,
            )
            processes.append(process)
            handles.append(handle)
            log_paths.append(log_path)
            time.sleep(0.4 if party else 0.8)

        deadline = time.monotonic() + timeout
        while any(process.poll() is None for process in processes):
            if time.monotonic() > deadline:
                for process in processes:
                    if process.poll() is None:
                        process.kill()
                raise TimeoutError(
                    f'PRAC parties exceeded {timeout} seconds'
                )
            time.sleep(0.25)

        returncodes = [process.returncode for process in processes]
        if any(code != 0 for code in returncodes):
            raise RuntimeError(f'PRAC party return codes: {returncodes}')
    finally:
        for handle in handles:
            handle.close()

    elapsed = time.perf_counter() - started
    outputs = [path.read_text(encoding='utf-8') for path in log_paths]
    combined = '\n\n'.join(
        f'=== party {party} ===\n{text}'
        for party, text in enumerate(outputs)
    )
    (prac_logs / 'combined.log').write_text(combined, encoding='utf-8')
    print(combined[-8000:])
    print(f'PRAC group elapsed={elapsed:.3f}s')
    return combined, elapsed

prac_summary_rows = []
prac_operation_rows = []

if RUN_EXACT_PRAC:
    try:
        if not (prac_dir / '.git').exists():
            run_bounded(
                f'git clone --branch popets-repro --depth 1 '
                f'{PRAC_URL} {prac_dir}',
                timeout=300,
                log_path=prac_logs / 'clone.log',
            )

        current, _ = run_bounded(
            'git rev-parse HEAD', cwd=prac_dir, timeout=30
        )
        current = current.strip().splitlines()[-1]
        if current != PRAC_COMMIT:
            run_bounded(
                f'git fetch --depth 1 origin {PRAC_COMMIT}',
                cwd=prac_dir,
                timeout=300,
            )
            run_bounded(
                f'git checkout --detach {PRAC_COMMIT}',
                cwd=prac_dir,
                timeout=60,
            )

        build_seconds = 0.0
        if not (prac_dir / 'prac').exists():
            _, build_seconds = run_bounded(
                f'make -j{PRAC_BUILD_JOBS}',
                cwd=prac_dir,
                timeout=1200,
                log_path=prac_logs / 'build.log',
            )
        else:
            print('Reusing compiled PRAC binary:', prac_dir / 'prac')

        commands = [
            f'./prac -o -t 1 0 duoram {PRAC_DEPTH} {PRAC_ITEMS}',
            f'./prac -o -t 1 1 localhost duoram '
            f'{PRAC_DEPTH} {PRAC_ITEMS}',
            f'./prac -o -t 1 2 localhost localhost duoram '
            f'{PRAC_DEPTH} {PRAC_ITEMS}',
        ]
        output, driver_seconds = run_party_group(
            commands, prac_dir, timeout=900
        )

        section_pattern = re.compile(
            r'=====\s*(.*?)\s*=====.*?'
            r'(\d+(?:\.\d+)?)\s+message bytes sent.*?'
            r'(\d+(?:\.\d+)?)\s+milliseconds? wall clock time',
            flags=re.I | re.S,
        )
        party_blocks = re.findall(
            r'=== party (\d+) ===\n(.*?)(?=\n\n=== party|\Z)',
            output,
            flags=re.S,
        )
        for party, party_text in party_blocks:
            for operation, sent, elapsed_ms in section_pattern.findall(
                party_text
            ):
                prac_operation_rows.append(dict(
                    party=int(party),
                    operation=operation.strip(),
                    communication_bytes=float(sent),
                    wall_time_ms=float(elapsed_ms),
                ))

        operation_df = pd.DataFrame(prac_operation_rows)
        if operation_df.empty:
            raise RuntimeError('PRAC output parser found no operations')

        protocol_seconds = (
            operation_df.groupby('party')['wall_time_ms'].sum().max()
            / 1000.0
        )
        communication_mb = (
            operation_df['communication_bytes'].sum() / 1e6
        )
        prac_summary_rows.append(dict(
            method='PRAC/Duoram',
            artifact_kind='official exact native artifact',
            native_operation='dependent/independent ORAM operations',
            n=2 ** PRAC_DEPTH,
            items_per_operation=PRAC_ITEMS,
            parties=3,
            protocol_seconds=protocol_seconds,
            driver_seconds=driver_seconds,
            communication_mb=communication_mb,
            build_seconds=build_seconds,
            commit=PRAC_COMMIT,
            status='success',
            directly_comparable_to_sle_transition=False,
        ))
    except Exception as exc:
        traceback.print_exc()
        prac_summary_rows.append(dict(
            method='PRAC/Duoram',
            artifact_kind='official exact native artifact',
            native_operation='dependent/independent ORAM operations',
            n=2 ** PRAC_DEPTH,
            items_per_operation=PRAC_ITEMS,
            parties=3,
            protocol_seconds=np.nan,
            driver_seconds=np.nan,
            communication_mb=np.nan,
            build_seconds=np.nan,
            commit=PRAC_COMMIT,
            status=f'failed: {type(exc).__name__}: {exc}',
            directly_comparable_to_sle_transition=False,
        ))
else:
    prac_summary_rows.append(dict(
        method='PRAC/Duoram', status='disabled',
        directly_comparable_to_sle_transition=False,
    ))

prac_exact_summary = pd.DataFrame(prac_summary_rows)
prac_exact_operations = pd.DataFrame(prac_operation_rows)
prac_exact_summary.to_csv(
    project / 'prac_exact_native_summary.csv', index=False
)
prac_exact_operations.to_csv(
    project / 'prac_exact_native_operations.csv', index=False
)
display(prac_exact_summary)
if len(prac_exact_operations):
    display(prac_exact_operations)


## Published SOTA context: adjacent tasks, not direct timings

The following table prevents a common paper error: comparing `3.2 s`
from one protocol directly with `4.9 s` from another protocol that uses
a different number of parties, hardware, security model, input
representation, and output functionality.

The table answers **what each paper demonstrates** and **why it is not
an interchangeable implementation of SLE's verified coupled epoch
transition**.


In [ ]:
# CELL 15. Primary-source literature comparison
literature = pd.DataFrame([
    dict(
        work='Mohassel, Rindal, Rosulek (CCS 2020)',
        task='Secret-shared database join',
        security_model='3-party honest majority; semi-honest',
        published_result='O(n), constant rounds; 1M-row join in 4.9 s',
        relation_to_sle=(
            'Closest generic join neighbor. Unique primary keys give '
            'the strongest leakage guarantee; not a coupled epoch '
            'transition with biometric multiplicities and rollback.'
        ),
        status='published',
        source='https://eprint.iacr.org/2019/518',
    ),
    dict(
        work='Chase, Ghosh, Poburinnaya (ASIACRYPT 2020)',
        task='Secret-shared random shuffle',
        security_model='2-party additive sharing; semi-honest',
        published_result='Communication-efficient online shuffle phase',
        relation_to_sle=(
            'Shuffle primitive hides row order but does not evolve '
            'equal locator values across two semantic structures or '
            'validate their multiplicity relation.'
        ),
        status='published',
        source=(
            'https://www.microsoft.com/en-us/research/'
            'publication/secret-shared-shuffle/'
        ),
    ),
    dict(
        work='Song et al. (NDSS 2024)',
        task='Maliciously secure secret-shared shuffle',
        security_model='2-party authenticated additive sharing; malicious',
        published_result=(
            'Reported more efficient than MP-SPDZ malicious SSS '
            'in the evaluated setting'
        ),
        relation_to_sle=(
            'Stronger adversary model than the current passive SLE '
            'prototype; still a shuffle primitive rather than a '
            'verified database-index epoch lifecycle.'
        ),
        status='published',
        source=(
            'https://www.ndss-symposium.org/ndss-paper/'
            'secret-shared-shuffle-with-malicious-security/'
        ),
    ),
    dict(
        work='GigaDORAM (USENIX Security 2023)',
        task='Distributed oblivious random access',
        security_model='3-server DORAM',
        published_result='More than 700 queries/s at 2^31 addresses',
        relation_to_sle=(
            'Query-access SOTA context. It protects access paths; it '
            'does not perform synchronized epoch refresh and relation '
            'validation across database and candidate index.'
        ),
        status='published',
        source=(
            'https://www.usenix.org/conference/usenixsecurity23/'
            'presentation/falk'
        ),
    ),
    dict(
        work='PRAC (PoPETs 2024)',
        task='3-party MPC dynamic data structures',
        security_model='3-party DORAM/MPC',
        published_result=(
            '>27x wall-clock and >3x bandwidth improvement for '
            'binary search at 2^26 items'
        ),
        relation_to_sle=(
            'Efficient oblivious query structures, not epoch '
            're-keying of duplicated hidden references.'
        ),
        status='published',
        source='https://doi.org/10.56553/popets-2024-0100',
    ),
    dict(
        work='Bifrost (2026)',
        task='Redundancy-free secure two-party join',
        security_model='2-party PSI plus oblivious shuffle',
        published_result=(
            '2.54-22.32x speedup and 84.15-88.97% less '
            'communication than iPrivJoin, up to 100 GB'
        ),
        relation_to_sle=(
            'Strong recent join context for separately owned '
            'datasets, but it does not provide epoch unlinkability, '
            'coupled-state validation, or atomic rollback.'
        ),
        status='preprint',
        source='https://arxiv.org/abs/2602.01225',
    ),
    dict(
        work='SLIDE (ePrint 2025/165)',
        task='Uniform Shamir-share shuffle',
        security_model='n-party Shamir; malicious honest majority',
        published_result='Claimed linear online communication',
        relation_to_sle=(
            'Potentially relevant primitive, but the ePrint record '
            'was withdrawn on 2026-06-26 and is excluded as an '
            'established SOTA baseline.'
        ),
        status='withdrawn',
        source='https://eprint.iacr.org/2025/165',
    ),
])
literature.to_csv(project / 'sle_published_sota_context.csv', index=False)
display(literature)


## Automatic interpretation with claim boundaries

The final cell prints only statements supported by completed rows.
Even if SLE wins the same-backend benchmark, the correct paper claim is
not "SLE is faster than every SOTA system." The defensible claim is that
SLE reduces the cost of the **specific verified coupled epoch-transition
task** relative to the reproduced generic alternatives under identical
MP-SPDZ conditions.


In [ ]:
# CELL 16. Generate numerical paper-ready comparisons
lines = []
paper_rows = []
for n in sorted(direct_ratios['n'].unique()):
    rows = direct_ratios[direct_ratios['n'] == n]
    sle = rows[rows['method'] == 'sle']
    if sle.empty:
        continue
    sle = sle.iloc[0]
    lines.append(f'N={n}: SLE transition passed relation verification.')
    for method in ('oram_reindex', 'sort_merge', 'pointer_join'):
        row = rows[rows['method'] == method]
        if row.empty:
            continue
        row = row.iloc[0]
        label = method_labels[method]
        paper_rows.append(dict(
            n=n,
            baseline=label,
            sle_seconds=sle.total_seconds,
            baseline_seconds=row.total_seconds,
            speedup_x=row.runtime_vs_sle,
            sle_global_mb=sle.global_mb,
            baseline_global_mb=row.global_mb,
            communication_reduction_x=row.communication_vs_sle,
            all_relation_checks_passed=True,
        ))
        lines.append(
            f'  versus {label}: {sle.total_seconds:.3f} s vs '
            f'{row.total_seconds:.3f} s ({row.runtime_vs_sle:.2f}x '
            f'speedup); {sle.global_mb:.3f} MB vs '
            f'{row.global_mb:.3f} MB '
            f'({row.communication_vs_sle:.2f}x less communication).'
        )

lines.append('')
lines.append(
    'Allowed claim: same-backend numerical advantage over measured '
    'SOTA-derived ORAM and oblivious-join/sort adaptations for the '
    'verified coupled epoch-transition task at completed N values.'
)
lines.append(
    'Not allowed: claim that the adaptations are byte-for-byte '
    'reproductions of CCS 2020, NDSS 2024, GigaDORAM, PRAC, or Bifrost.'
)
if len(prac_exact_summary):
    prac_row = prac_exact_summary.iloc[0]
    lines.append('')
    if prac_row.get('status') == 'success':
        lines.append(
            'Exact PRAC/Duoram native artifact: '
            f"N={int(prac_row['n'])}, "
            f"protocol time={prac_row['protocol_seconds']:.3f} s, "
            f"communication={prac_row['communication_mb']:.3f} MB. "
            'This is native-operation context, not a direct SLE ratio.'
        )
    else:
        lines.append(
            'Exact PRAC/Duoram native artifact status: '
            f"{prac_row.get('status')}"
        )

paper_comparison = pd.DataFrame(paper_rows)
paper_comparison.to_csv(
    project / 'sle_paper_ready_numerical_comparison.csv', index=False
)
display(paper_comparison)

report_text = '\n'.join(lines)
print(report_text)
(project / 'sle_sota_interpretation.txt').write_text(
    report_text, encoding='utf-8'
)


In [ ]:
# CELL 17. Bundle every result needed for analysis and the paper
bundle = project / 'sle_sota_fast_reproducible_results.zip'
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
    result_files = [
        'sle_sota_direct_raw.csv',
        'sle_sota_direct_summary.csv',
        'sle_sota_direct_ci95.csv',
        'sle_sota_direct_ratios.csv',
        'sle_paper_ready_numerical_comparison.csv',
        'sle_sota_configurations.csv',
        'sle_sota_same_backend_comparison.png',
        'sle_published_sota_context.csv',
        'sle_sota_interpretation.txt',
        'prac_exact_native_summary.csv',
        'prac_exact_native_operations.csv',
        'SLE_FORMAL_PROTOCOL.md',
        'SLE_SOTA_NUMERICAL_COMPARISON_RU.md',
    ]
    for name in result_files:
        path = project / name
        if path.exists():
            archive.write(path, arcname=name)

    for name in program_names:
        path = project / 'Programs/Source' / name
        archive.write(path, arcname=f'Programs/Source/{name}')

    if prac_logs.exists():
        for path in prac_logs.rglob('*'):
            if path.is_file():
                archive.write(
                    path,
                    arcname=f'logs/prac_exact/{path.name}',
                )

    manifest = dict(
        generated_at_utc=pd.Timestamp.utcnow().isoformat(),
        mp_spdz_commit=MP_SPDZ_COMMIT,
        players=PLAYERS,
        threshold=THRESHOLD,
        active_n=ACTIVE_N,
        repeats=REPEATS,
        paper_mode=PAPER_MODE,
        root_bits=ROOT_BITS,
        block_bits=BLOCK_BITS,
        exact_prac_enabled=RUN_EXACT_PRAC,
        exact_prac_commit=PRAC_COMMIT,
    )
    manifest_path = project / 'reproducibility_manifest.json'
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    archive.write(manifest_path, arcname=manifest_path.name)

print('Saved:', bundle)
try:
    from google.colab import files
    files.download(str(bundle))
except ImportError:
    print('Download is available only in Colab')
